<a href="https://colab.research.google.com/github/mnsbharadwaj/AI-NLP/blob/master/Sentiment_classification_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# This script demonstrates a simple transfer learning model for sentiment analysis.
# It uses the Hugging Face `transformers` library to fine-tune a pre-trained
# BERT model on a well-known sentiment dataset from the `datasets` library.

from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments
import torch

# Define the model and tokenizer to use. We'll use a pre-trained BERT model.
MODEL_NAME = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

# --- 1. Data Preparation ---
# Load a sentiment analysis dataset directly from the Hugging Face hub.
# We'll use the 'imdb' dataset, which is a classic for this task.
print("Loading IMDB dataset...")
# Load a very small portion for a quick demo. This will make training much faster.
dataset = load_dataset("imdb", split="train[:100]")

# The dataset is already split into 'train' and 'test', so we'll use a small
# portion of the training set for our fine-tuning and validation.
dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset['train']
val_dataset = dataset['test']

# --- 2. Tokenization and Dataset Class ---
# This class will be passed to the Trainer.
class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

    def __getitem__(self, idx):
        # Tokenize each text individually within the __getitem__ method
        # and return the required format for the model.
        tokenized_output = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=512,
            return_tensors='pt'
        )

        # Squeeze to remove the batch dimension that huggingface adds.
        tokenized_output = {k: v.squeeze() for k, v in tokenized_output.items()}

        # Add the label
        tokenized_output['labels'] = torch.tensor(self.labels[idx])

        return tokenized_output

    def __len__(self):
        return len(self.texts)

# Create the custom datasets with the encodings and labels.
train_dataset = SentimentDataset(train_dataset['text'], train_dataset['label'], tokenizer)
val_dataset = SentimentDataset(val_dataset['text'], val_dataset['label'], tokenizer)

# --- 3. Model Loading and Training ---
# Load the pre-trained BERT model for sequence classification.
# `num_labels` is set to 2 for our binary classification (positive/negative).
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# Get the initial weights of the classifier layer before training.
print("--- Initial Classifier Weights (before training) ---")
print(model.classifier.weight)

# Define the training arguments. These settings configure how the model will be trained.
training_args = TrainingArguments(
    output_dir='./results',          # directory to save model checkpoints
    num_train_epochs=3,              # total number of training epochs
    per_device_train_batch_size=8,   # batch size per device during training
    per_device_eval_batch_size=8,    # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for storing logs
    logging_steps=10,
    report_to="tensorboard"          # Only report to TensorBoard
)

# Initialize the Trainer. This object handles the training loop.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# Start the training process.
print("\nStarting fine-tuning...")
trainer.train()
print("Fine-tuning complete.")

# Get the updated weights of the classifier layer after training.
print("\n--- Final Classifier Weights (after training) ---")
print(model.classifier.weight)

# --- 4. Prediction and Inference ---
# Example of how to use the fine-tuned model for prediction.
def predict_sentiment(text):
    # Tokenize the input text.
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True)

    # Get the model's output (logits).
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    # The sentiment is the index of the highest logit.
    # 0 for negative, 1 for positive.
    prediction = torch.argmax(logits, dim=1).item()

    # Return a human-readable result.
    return "Positive" if prediction == 1 else "Negative"

print("\n--- Making predictions ---")
new_text = "The service was a little slow, but the food was fantastic."
sentiment = predict_sentiment(new_text)
print(f"Text: '{new_text}'")
print(f"Predicted sentiment: {sentiment}")

new_text_2 = "I'm so disappointed with the quality."
sentiment_2 = predict_sentiment(new_text_2)
print(f"Text: '{new_text_2}'")
print(f"Predicted sentiment: {sentiment_2}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading IMDB dataset...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


--- Initial Classifier Weights (before training) ---
Parameter containing:
tensor([[-0.0146,  0.0357, -0.0114,  ..., -0.0395, -0.0142,  0.0524],
        [ 0.0142, -0.0027,  0.0032,  ...,  0.0118,  0.0104, -0.0158]],
       requires_grad=True)

Starting fine-tuning...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,0.680300
20,0.605800
30,0.443000


Fine-tuning complete.

--- Final Classifier Weights (after training) ---
Parameter containing:
tensor([[-0.0146,  0.0357, -0.0115,  ..., -0.0395, -0.0143,  0.0524],
        [ 0.0143, -0.0027,  0.0032,  ...,  0.0119,  0.0105, -0.0159]],
       requires_grad=True)

--- Making predictions ---
Text: 'The service was a little slow, but the food was fantastic.'
Predicted sentiment: Negative
Text: 'I'm so disappointed with the quality.'
Predicted sentiment: Negative
